In [34]:
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv

In [35]:
load_dotenv()

True

In [36]:

# this part is only ai generated part

from langchain_community.document_loaders import DirectoryLoader, PDFPlumberLoader
from langchain_core.documents import Document

book_folder = "./books"

loader = DirectoryLoader(
    book_folder,
    glob="*.pdf",
    loader_cls=PDFPlumberLoader,
    show_progress=True
)

pages = loader.load()

documents = []

# Group pages by PDF
books = {}

for page in pages:
    source = page.metadata["source"]
    books.setdefault(source, []).append(page)

# Create one Document for each PDF
for source, book_pages in sorted(books.items()):
    full_text = "\n\n".join(
        page.page_content for page in book_pages
    )

    documents.append(
        Document(
            page_content=full_text,
            metadata={
                "source": source,
                "file_name": source.split("\\")[-1],
                "total_pages": len(book_pages)
            }
        )
    )

print("Total documents:", len(documents))



100%|██████████| 3/3 [00:51<00:00, 17.22s/it]

Total documents: 3


In [37]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [38]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

In [39]:
chunks=splitter.split_documents(documents)

In [40]:
len(chunks)

2789

In [41]:
from langchain_huggingface import HuggingFaceEmbeddings

emb_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11153.91it/s]


In [42]:
from langchain_community.vectorstores import FAISS

vector_store=FAISS.from_documents(
    chunks,
    emb_model,
)

In [43]:
retriver=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)

In [44]:
retriver.invoke("what is linear regression?")

[Document(id='3fab182f-3335-4371-90a4-d8d3a351f186', metadata={'source': 'books\\book3.pdf', 'file_name': 'book3.pdf', 'total_pages': 392}, page_content='Linear Models\nLinear models are a class of models that are widely used in practice and have been\nstudied extensively in the last few decades, with roots going back over a hundred\nyears. Linear models make a prediction using a linear function of the input features,\nwhich we will explain shortly.\nLinear models for regression\nFor regression, the general prediction formula for a linear model looks as follows:\nŷ = w[0] * x[0] + w[1] * x[1] + ... + w[p] * x[p] + b\nHere, x[0] to x[p] denotes the features (in this example, the number of features is p)\nof a single data point, w and b are parameters of the model that are learned, and ŷ is\nthe prediction the model makes. For a dataset with a single feature, this is:\nŷ = w[0] * x[0] + b\nwhich you might remember from high school mathematics as the equation for a line.\nHere, w[0] is th

In [45]:
from langchain_core.prompts import PromptTemplate

prompt=PromptTemplate(
    template="""you are a helpful machine learning ai assistsnt
    please answer the question \n {question}
    from the following content \n {content}
    """,
    input_variables=["question","content"]
)

In [46]:
from langchain_groq import ChatGroq

model=ChatGroq(model="openai/gpt-oss-120b")

In [47]:
def format_docs(retrieved_docs):
    context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
    )
    return context

In [48]:
from langchain_core.runnables import RunnablePassthrough,RunnableParallel,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

parser=StrOutputParser()

In [49]:
chain=({
     "question":RunnablePassthrough(),
     "content":retriver | RunnableLambda(format_docs)
} | prompt | model | parser )


In [50]:
query="what is linear regression in short?"


result=chain.invoke(query)

result

'**Linear regression (ordinary least squares)** is a basic linear model for regression that predicts a target\u202f\\(y\\) by computing a weighted sum of the input features plus a bias (intercept):\n\n\\[\n\\hat{y}= w_0 x_0 + w_1 x_1 + \\dots + w_p x_p + b\n\\]\n\nThe model’s parameters \\(w\\) (the feature weights) and \\(b\\) (the bias) are chosen to **minimize the mean‑squared error** between the predictions \\(\\hat{y}\\) and the true target values on the training data. In other words, it finds the line (or hyper‑plane in higher dimensions) that best fits the data in the least‑squares sense.'